# RSNA Knee — M2 training on Kaggle GPUs (one notebook per training session)

**Settings:** Accelerator **GPU T4 x2** · Internet **off** · Persistence: none.
**Inputs:** notebooks `rsna-knee-prep-chunk0` + `rsna-knee-prep-chunk1` (Your Work → Notebooks), dataset `rsna-knee-train-inputs` (labels, M1 start checkpoints, DINOv2 weights), and — only when a job uses `--train-labels pseudo_*.parquet` — the output of `rsna-knee-score`. Do **not** attach `rsna-knee-models`.

**One notebook per session** (`rsna-knee-train`, `rsna-knee-train-s2`, `-s3`, …): a notebook's output is replaced by every new version, so earlier runs would disappear. Start a session with *Copy & Edit* of the previous one, rename it, *File → Import Notebook* if the code changed, set `JOBS`.

1. Set `COMMON` and `JOBS` in the next cell. Every (run, fold) pair is one task; tasks go to whichever GPU is free, in list order, so two experiments can share a session (e.g. fold 0 of two ideas).
2. `MODE = "benchmark"` (~15 min) measures the first tasks and estimates the session length. `MODE = "train"` → **Save Version → Save & Run All (Commit)**. Runs in the background up to 12 h.
3. Score / blend / package: attach this notebook's output to `rsna-knee-score`.

Generated by `scripts/build_train_notebook.py` — edit the repo, not this notebook.

In [ ]:
import os, sys, time
T_START = time.time()

MODE = "benchmark"            # "benchmark" (~15 min speed test of the first jobs) or "train"
# Flags shared by every job (the ft3a recipe). A job's own flags come after these and win.
COMMON = "--epochs 12 --frozen-blocks 3 --eval-every 6 --bs 4 --accum 4"
JOBS = [                      # (run name, folds, extra flags) -> /kaggle/working/m2/<run>/
    ("ft4a", [0], "--frozen-blocks 0 --bs 2 --accum 8"),   # all 12 blocks trained
    ("ft4p", [0], "--train-labels pseudo_pl1.parquet"),    # pseudo-labels from rsna-knee-score
]
WORKERS_PER_GPU = 2           # DataLoader workers per process (Kaggle has 4 CPU cores)
TIME_BUDGET_H = 11.3          # don't start a task expected to end after this (limit: 12 h)
BENCH_MIN = 12                # benchmark: minutes of training per GPU
MONITOR_MIN = 10              # train mode: GPU memory / utilisation line every N minutes

In [ ]:
import glob, shutil, subprocess

def find(predicate, top="/kaggle/input", max_depth=7):
    hits = []
    top_depth = top.rstrip("/").count("/")
    for root, dirs, files in os.walk(top):
        dirs[:] = [d for d in dirs if d not in {"train_series", "test_series"}]
        if root.count("/") - top_depth >= max_depth:
            dirs[:] = []
        if predicate(root, files):
            hits.append(root)
    return hits

LABEL_DIR = find(lambda r, f: "soft_labels.parquet" in f and "folds.csv" in f)
MODEL_DIR = [d for d in find(lambda r, f: "m1" in os.listdir(r) and "dinov2" in os.listdir(r))]
MODEL_DIR.sort(key=lambda d: not LABEL_DIR or d != LABEL_DIR[0])    # prefer rsna-knee-train-inputs
TAR_DIRS = find(lambda r, f: any(x.startswith("shards_c") and x.endswith(".tar") for x in f))
TARS = sorted(t for d in TAR_DIRS for t in glob.glob(f"{d}/shards_c*_p*.tar"))
assert MODEL_DIR, "no folder with m1/ and dinov2/ - attach rsna-knee-train-inputs (scripts/make_train_inputs.py)"
assert LABEL_DIR, "rsna-knee-train-inputs not found - Add Input -> Datasets"
assert TARS, "shard tars not found - Add Input -> Notebooks -> rsna-knee-prep-chunk0 / chunk1"
MODEL_DIR, LABEL_DIR = MODEL_DIR[0], LABEL_DIR[0]
M1_DIR, WEIGHTS = f"{MODEL_DIR}/m1", f"{MODEL_DIR}/dinov2/model.safetensors"
assert glob.glob(f"{M1_DIR}/fold*.pt"), f"no m1/fold*.pt in {MODEL_DIR}"
assert os.path.exists(WEIGHTS), f"no dinov2/model.safetensors in {MODEL_DIR}"
print("models:", MODEL_DIR, "| labels:", LABEL_DIR, f"| {len(TARS)} shard tars")
if len(TARS) != 12:
    print("WARNING: expected 12 tars (6 per prep notebook) - is one prep notebook missing?")

import torch, timm
GPUS = list(range(torch.cuda.device_count()))
print("python", sys.version.split()[0], "| torch", torch.__version__, "| timm", timm.__version__,
      "| CPUs", os.cpu_count(), "| GPUs", [torch.cuda.get_device_name(g) for g in GPUS])
assert GPUS, "no GPU - set Accelerator to GPU T4 x2"
if len(GPUS) == 1:
    print("NOTE: only 1 GPU - folds run one after another (choose GPU T4 x2 for two at once)")

In [ ]:
from concurrent.futures import ThreadPoolExecutor

CODE, WORK = "/tmp/code", "/tmp/work"
OUT_ROOT = "/kaggle/working/m2"
SHARDS = f"{WORK}/data/shards"
for d in (f"{CODE}/src/models", SHARDS, f"{WORK}/data/labels", f"{WORK}/runs/m1", OUT_ROOT):
    os.makedirs(d, exist_ok=True)
for f in ("src/__init__.py", "src/models/__init__.py"):          # empty package markers
    open(f"{CODE}/{f}", "w").close()

# labels, M1 checkpoints (head init) and the output folder, in the layout m2.py expects
for f in ("soft_labels.parquet", "folds.csv"):
    shutil.copy(f"{LABEL_DIR}/{f}", f"{WORK}/data/labels/{f}")
for link, target in ((f"{WORK}/runs/m1/v2", M1_DIR), (f"{WORK}/runs/m2", OUT_ROOT)):
    if not os.path.islink(link):
        os.symlink(target, link)

tmpfs = any(l.split()[1] == "/tmp" and l.split()[2] == "tmpfs" for l in open("/proc/mounts"))
assert not tmpfs, "/tmp is in RAM here - the shards would not fit"
free = shutil.disk_usage("/tmp").free / 1e9
print(f"/tmp free: {free:.0f} GB (shards need ~24 GB)")
assert free > 26, "not enough disk in /tmp for the shards"

t0 = time.time()
def untar(t):
    return subprocess.run(["tar", "-xf", t, "-C", SHARDS], capture_output=True, text=True)
with ThreadPoolExecutor(4) as ex:
    bad = [r.stderr for r in ex.map(untar, TARS) if r.returncode != 0]
n = len(glob.glob(f"{SHARDS}/*.npz"))
print(f"unpacked {n} studies in {(time.time() - t0) / 60:.1f} min" + (f" - ERRORS: {bad[:2]}" if bad else ""))
assert n >= 4400, "shards incomplete"

# pseudo-label files (labels/pseudo_*.parquet in the rsna-knee-score output), for --train-labels
pseudo = {}
for d in find(lambda r, f: any(x.startswith("pseudo_") and x.endswith(".parquet") for x in f)):
    for f in glob.glob(f"{d}/pseudo_*.parquet"):
        pseudo.setdefault(os.path.basename(f), []).append(f)
for name, fs in sorted(pseudo.items()):
    shutil.copy(fs[0], f"{WORK}/data/labels/{name}")
    print(f"pseudo labels: {name} from {fs[0]}" + (f" (also in {len(fs) - 1} other input(s))" if len(fs) > 1 else ""))


## Source files (embedded from the repo)

In [ ]:
%%writefile /tmp/code/src/models/data.py
"""Pack the per-study DINOv2 embeddings into a few arrays for fast training.

    python -m src.models.data            # builds data/embeddings/<tag>/cache_g2/ once

Cache layout (all rows in the order of uids.txt):
    feats.npy    float16 [N, 6, 32, 768]        CLS + mean patch per slice
    grid.npy     float16 [N, 6, 32, g*g, 384]   patch grid pooled 4x4 -> g x g (g=2 default)
    present.npy  bool    [N, 6]                 which slots exist for the study
Slots: SLOTS below. Mirrored slots (cor_fl_m, ax_fl_m) hold the left-right flipped
images; sagittal kinds are mirrored by reversing the slice order instead.
"""
from __future__ import annotations

import argparse
import time
from pathlib import Path

import numpy as np

SLOTS = ("sag_fl", "cor_fl", "cor_fl_m", "ax_fl", "ax_fl_m", "sag_t1")
KINDS = ("sag_fl", "cor_fl", "ax_fl", "sag_t1")
# slot index of each kind, normal and mirrored
KIND_SLOT = np.array([0, 1, 3, 5])
KIND_SLOT_MIRROR = np.array([0, 2, 4, 5])
SAGITTAL = np.array([True, False, False, True])      # mirrored by reversing slice order
N_SLICES, FEAT_DIM, PATCH_DIM = 32, 768, 384


MIRROR_IMAGE = ~SAGITTAL           # mirror: coronal/axial flip the image, sagittal reverses slices
SIZE = 256


def augment_series(vol: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Same random affine + intensity change for every slice of one series (uint8 [S,H,W])."""
    import cv2
    s = rng.uniform(0.9, 1.1)
    ang = rng.uniform(-10, 10)
    tx, ty = rng.uniform(-0.06, 0.06, 2) * SIZE
    m = cv2.getRotationMatrix2D((SIZE / 2, SIZE / 2), ang, s)
    m[:, 2] += (tx, ty)
    out = np.empty_like(vol)
    for i in range(len(vol)):
        out[i] = cv2.warpAffine(vol[i], m, (SIZE, SIZE), flags=cv2.INTER_LINEAR,
                                borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    a, b = rng.uniform(0.85, 1.15), rng.uniform(-12, 12)
    return np.clip(out.astype(np.float32) * a + b, 0, 255).astype(np.uint8)


def sample_study(arrays: dict, train: bool, rng: np.random.Generator | None = None,
                 n_slices: int = 16, mirror: bool = False, p_mirror: float = 0.5,
                 p_series: float = 0.2):
    """kneeprep arrays -> (vol uint8 [4, S, 256, 256], present bool [4], positions int [S]).

    train: every (32/S)-th slice with a random offset, random mirror, series dropout and
    augmentation. eval: all 32 slices, mirrored only if `mirror`.
    """
    if train:
        step = N_SLICES // n_slices
        idx = rng.integers(0, step) + step * np.arange(n_slices)
        mirror = rng.random() < p_mirror
    else:
        idx = np.arange(N_SLICES)
    present = np.array([k in arrays for k in KINDS])
    if train and p_series > 0 and present.sum() > 1:
        drop = (rng.random(len(KINDS)) < p_series) & present
        if (present & ~drop).any():
            present &= ~drop
    vol = np.zeros((len(KINDS), len(idx), SIZE, SIZE), np.uint8)
    for k, kind in enumerate(KINDS):
        if not present[k]:
            continue
        v = arrays[kind]
        if mirror:
            v = v[:, :, ::-1] if MIRROR_IMAGE[k] else v[::-1]
        v = np.ascontiguousarray(v[idx])
        vol[k] = augment_series(v, rng) if train else v
    return vol, present, idx.astype(np.int64)


def pool_grid(grid: np.ndarray, g_out: int) -> np.ndarray:
    """[S, G*G, D] (row-major G x G) -> [S, g_out*g_out, D] by block averaging."""
    s, n, d = grid.shape
    g_in = int(round(n ** 0.5))
    if g_out == g_in:
        return grid
    f = g_in // g_out
    assert f * g_out == g_in, f"cannot pool {g_in}x{g_in} to {g_out}x{g_out}"
    x = grid.astype(np.float32).reshape(s, g_out, f, g_out, f, d).mean((2, 4))
    return x.reshape(s, g_out * g_out, d)


def build_cache(emb_dir: Path, g_out: int = 2) -> Path:
    files = sorted(p for p in emb_dir.glob("*.npz"))
    if not files:
        raise SystemExit(f"no embeddings in {emb_dir}")
    out = emb_dir / f"cache_g{g_out}"
    out.mkdir(exist_ok=True)
    n = len(files)
    feats = np.lib.format.open_memmap(out / "feats.tmp.npy", "w+", np.float16,
                                      (n, len(SLOTS), N_SLICES, FEAT_DIM))
    grid = np.lib.format.open_memmap(out / "grid.tmp.npy", "w+", np.float16,
                                     (n, len(SLOTS), N_SLICES, g_out * g_out, PATCH_DIM))
    present = np.zeros((n, len(SLOTS)), bool)
    t0 = time.time()
    for i, f in enumerate(files):
        with np.load(f) as z:
            for j, slot in enumerate(SLOTS):
                if slot in z.files:
                    feats[i, j] = z[slot]
                    grid[i, j] = pool_grid(z[f"{slot}_grid"], g_out)
                    present[i, j] = True
                else:
                    feats[i, j] = 0
                    grid[i, j] = 0
        if (i + 1) % 500 == 0 or i + 1 == n:
            print(f"  {i + 1}/{n}  {time.time() - t0:.0f}s")
    feats.flush()
    grid.flush()
    del feats, grid
    (out / "feats.tmp.npy").replace(out / "feats.npy")
    (out / "grid.tmp.npy").replace(out / "grid.npy")
    np.save(out / "present.npy", present)
    (out / "uids.txt").write_text("\n".join(f.stem for f in files))
    print(f"cache: {n} studies -> {out}")
    print("slots present:", dict(zip(SLOTS, present.sum(0).tolist())))
    return out


def channel_stats(cache: dict, chunk: int = 64) -> dict[str, np.ndarray]:
    """Per-channel mean / std over every present slice (no labels involved).

    feats: 768 channels; grid: 384 channels shared by all grid cells. DINOv2 has a few
    channels with very large values; standardising each channel stops them dominating.
    """
    feats, grid, present = cache["feats"], cache["grid"], cache["present"]
    sums = {k: np.zeros(d, np.float64) for k, d in (("f", feats.shape[-1]), ("g", grid.shape[-1]))}
    sq = {k: np.zeros_like(v) for k, v in sums.items()}
    n = {"f": 0, "g": 0}
    for s in range(0, len(feats), chunk):
        m = present[s:s + chunk]
        f = feats[s:s + chunk][m].astype(np.float64).reshape(-1, feats.shape[-1])
        g = grid[s:s + chunk][m].astype(np.float64).reshape(-1, grid.shape[-1])
        for key, a in (("f", f), ("g", g)):
            sums[key] += a.sum(0)
            sq[key] += (a * a).sum(0)
            n[key] += len(a)
    out = {}
    for key in ("f", "g"):
        mu = sums[key] / n[key]
        sd = np.sqrt(np.maximum(sq[key] / n[key] - mu * mu, 1e-12))
        out[f"{key}_mean"] = mu.astype(np.float32)
        out[f"{key}_std"] = np.maximum(sd, 1e-3).astype(np.float32)
    return out


def load_stats(cache_dir: Path, cache: dict) -> dict[str, np.ndarray]:
    path = cache_dir / "stats.npz"
    if path.exists():
        with np.load(path) as z:
            return {k: z[k] for k in z.files}
    stats = channel_stats(cache)
    np.savez(path, **stats)
    return stats


def load_cache(cache_dir: Path, in_memory: bool = True) -> dict:
    mode = None if in_memory else "r"
    return dict(
        uids=cache_dir.joinpath("uids.txt").read_text().split("\n"),
        feats=np.load(cache_dir / "feats.npy", mmap_mode=mode),
        grid=np.load(cache_dir / "grid.npy", mmap_mode=mode),
        present=np.load(cache_dir / "present.npy"),
    )


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", type=Path, default=Path("."))
    ap.add_argument("--tag", default="dinov2s_224_g4")
    ap.add_argument("--grid", type=int, default=2)
    args = ap.parse_args()
    build_cache(args.root / "data" / "embeddings" / args.tag, args.grid)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/code/src/models/folds.py
"""5-fold CV split over all training studies, grouped by patient.

    python -m src.models.folds

* Patient grouping: PatientID from data/raw/study_meta_chunk*.csv; all studies of a
  patient go to the same fold. Prints how many patients have more than one study.
* Stratification (greedy, deterministic): the 12 labels (soft label >= 0.5 counts as
  positive) plus the report language, so every fold has a similar mix of findings and
  reporting sites. The 58 human-labelled studies are spread across folds too.

Output: data/labels/folds.csv  StudyInstanceUID, PatientID, fold, lang, source
"""
from __future__ import annotations

import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

_LANG_HINTS = {
    "es": r"\b(menisco|derrame|rotura|ligamento|rodilla|sin|con|del)\b",
    "de": r"\b(meniskus|kreuzband|erguss|kniegelenk|und|der|die|kein)\b",
    "hr": r"\b(menisk[a-z]*|ligament[a-z]*|zglob[a-z]*|uredn[a-z]*|bez|uz)\b|[čćđ]",
    "nl": r"\b(van|het|een|geen|gewricht[a-z]*|kniegewricht)\b",
    "fr": r"\b(ménisque|épanchement|genou|du|des|pas)\b",
    "tr": r"[ığş]|\b(menisküs|bağ|izlendi|mevcuttur|normaldir)\b",
}


def guess_language(text: str) -> str:
    """Rough report-language guess (used only to balance folds)."""
    t = str(text).lower()
    if re.search(r"[а-яё]", t):
        return "bg"
    if re.search(r"[α-ω]", t):
        return "el"
    scores = {lang: len(re.findall(pat, t)) for lang, pat in _LANG_HINTS.items()}
    lang, best = max(scores.items(), key=lambda kv: kv[1])
    en = len(re.findall(r"\b(the|of|and|is|with|no|meniscus|ligament|effusion)\b", t))
    return lang if best > max(2, en) else "en"


def read_labels(path: Path) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype={"StudyInstanceUID": str})


def load_patients(raw: Path) -> pd.Series | None:
    files = sorted(raw.glob("study_meta_chunk*.csv"))
    if not files:
        return None
    meta = pd.concat([pd.read_csv(f, usecols=["StudyInstanceUID", "PatientID"], dtype=str)
                      for f in files])
    return meta.drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID")["PatientID"]


def greedy_group_folds(Y: np.ndarray, groups: np.ndarray, k: int, seed: int) -> np.ndarray:
    """Assign each group to a fold, keeping every column's positives and the fold
    sizes as even as possible. Rare columns are placed first."""
    G = groups.max() + 1
    gY = np.zeros((G, Y.shape[1]))
    np.add.at(gY, groups, Y)
    gsize = np.bincount(groups, minlength=G).astype(float)
    totals = Y.sum(0)
    tgt = totals / k
    tgt_size = len(groups) / k
    w = 1.0 / np.maximum(tgt, 1.0)

    rng = np.random.default_rng(seed)
    with np.errstate(divide="ignore"):
        rarity = np.where(gY > 0, totals[None, :], np.inf).min(1)   # rarest column present
    order = np.lexsort((rng.random(G), -gsize, rarity))

    cnt = np.zeros((k, Y.shape[1]))
    size = np.zeros(k)
    fold_of_group = np.empty(G, dtype=int)
    for g in order:
        before = (w * (cnt - tgt) ** 2).sum(1) + ((size - tgt_size) / tgt_size) ** 2 * 50
        after = (w * (cnt + gY[g] - tgt) ** 2).sum(1) + ((size + gsize[g] - tgt_size) / tgt_size) ** 2 * 50
        f = int(np.argmin(after - before))
        fold_of_group[g] = f
        cnt[f] += gY[g]
        size[f] += gsize[g]
    return fold_of_group[groups]


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", type=Path, default=Path("."))
    ap.add_argument("--labels", type=Path, default=None)
    ap.add_argument("--k", type=int, default=5)
    ap.add_argument("--seed", type=int, default=2026)
    args = ap.parse_args()

    raw = args.root / "data" / "raw"
    lab_path = args.labels or args.root / "data" / "labels" / "soft_labels.parquet"
    labels = read_labels(lab_path)
    train = pd.read_csv(raw / "train.csv", usecols=["StudyInstanceUID", "Report"])
    df = labels.merge(train, on="StudyInstanceUID", how="left")
    df = df[df["source"] != "missing"].reset_index(drop=True)

    pids = load_patients(raw)
    if pids is None:
        print("WARNING: no study_meta_chunk*.csv in data/raw -> no patient grouping")
        df["PatientID"] = ""
    else:
        df["PatientID"] = df["StudyInstanceUID"].map(pids).fillna("")
    key = df["PatientID"].where(df["PatientID"] != "", "study:" + df["StudyInstanceUID"])
    groups = pd.factorize(key)[0]
    per_patient = pd.Series(groups).value_counts()
    print(f"studies {len(df)} · patients {groups.max() + 1} · "
          f"patients with >1 study: {(per_patient > 1).sum()} "
          f"(covering {per_patient[per_patient > 1].sum()} studies) · "
          f"missing PatientID: {(df['PatientID'] == '').sum()}")

    df["lang"] = df["Report"].map(guess_language)
    Yl = (df[LABELS].to_numpy() >= 0.5).astype(float)
    Ylang = pd.get_dummies(df["lang"]).to_numpy().astype(float)
    Yhuman = (df["source"] == "human").to_numpy()[:, None].astype(float)
    Y = np.hstack([Yl, Ylang, Yhuman])
    df["fold"] = greedy_group_folds(Y, groups, args.k, args.seed)

    out = args.root / "data" / "labels" / "folds.csv"
    df[["StudyInstanceUID", "PatientID", "fold", "lang", "source"]].to_csv(out, index=False)

    summary = pd.DataFrame(Yl, columns=LABELS).groupby(df["fold"]).sum().astype(int)
    summary.insert(0, "studies", df.groupby("fold").size())
    summary["human"] = df.groupby("fold")["source"].apply(lambda s: (s == "human").sum())
    with pd.option_context("display.width", 200, "display.max_columns", 30):
        print("\npositives (>= 0.5) per fold:")
        print(summary.to_string())
        print("\nlanguages per fold:")
        print(pd.crosstab(df["fold"], df["lang"]).to_string())
    if pids is not None:
        leak = df.groupby("PatientID")["fold"].nunique()
        leak = leak[(leak > 1) & (leak.index != "")]
        print(f"\npatients split across folds: {len(leak)} (must be 0)")
    print(f"\nsaved -> {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/code/src/models/metrics.py
"""Out-of-fold scoring for the 12 labels.

Two views, reported side by side:
  auc_llm   : LLM-labelled studies; per label the 'uncertain' (0.5) rows are left out
              and the rest are 1/0. Main signal (thousands of studies).
  auc_human : the 58 human-labelled studies (true labels). Small and noisy, but the
              only check against the organisers' own definitions.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]


def safe_auc(y: np.ndarray, s: np.ndarray) -> float:
    y = np.asarray(y)
    if len(y) == 0 or y.min() == y.max():
        return float("nan")
    return float(roc_auc_score(y, s))


def score(pred: np.ndarray, target: np.ndarray, source: np.ndarray) -> pd.DataFrame:
    """pred, target: [n, 12]; source: 'llm' / 'human' per row."""
    rows = []
    llm = source == "llm"
    hum = source == "human"
    for j, lab in enumerate(LABELS):
        t, p = target[:, j], pred[:, j]
        certain = llm & (t != 0.5)
        rows.append(dict(
            label=lab,
            auc_llm=safe_auc(t[certain] >= 0.5, p[certain]),
            pos_llm=int((t[certain] >= 0.5).sum()),
            auc_human=safe_auc(t[hum] >= 0.5, p[hum]),
            pos_human=int((t[hum] >= 0.5).sum()),
        ))
    df = pd.DataFrame(rows).set_index("label")
    df.loc["MACRO"] = [df["auc_llm"].mean(), df["pos_llm"].sum(),
                       df["auc_human"].mean(), df["pos_human"].sum()]
    return df


In [ ]:
%%writefile /tmp/code/src/models/m1.py
"""M1: frozen DINOv2-S slice features -> small transformer aggregator -> 12 labels.

    python -m src.models.m1 --smoke          # 1 fold, 2 epochs, 400 studies: pipeline check
    python -m src.models.m1 --name base      # 5-fold CV, OOF scores, checkpoints

Needs: data/labels/soft_labels.parquet, data/labels/folds.csv (src.models.folds) and
the feature cache (src.models.data). Outputs go to runs/m1/<name>/:
    fold{k}.pt, oof.csv, metrics.csv, config.json, train.log

Per study the input is 4 series kinds x 32 slices = 128 tokens; each token is the
slice's CLS + mean-patch feature (768) plus, by default, its 2x2 patch grid (1536).
token = MLP(feature) + kind embedding + slice-position embedding
-> 2-layer transformer -> 12 learned label queries attend over the tokens -> 12 logits.

Inputs are standardised per channel (stats.npz in the cache dir, also saved in every
checkpoint). A masked mean of the tokens is added to each label's attention output, so
the model learns from the first epoch instead of waiting for the attention to focus.

Training: BCE on soft labels (1 / 0.5 / 0), AdamW + cosine, and augmentation:
  * mirror the knee left<->right (p=0.5): coronal/axial use the flipped-image
    features, sagittal kinds get their slice order reversed;
  * series dropout (drop whole kinds, keep >= 1) and slice dropout.
Validation/prediction average the normal and mirrored knee (TTA).
"""
from __future__ import annotations

import argparse
import json
import logging
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from .data import KIND_SLOT, KIND_SLOT_MIRROR, N_SLICES, SAGITTAL, load_cache, load_stats
from .folds import read_labels
from .metrics import LABELS, score

N_KINDS = 4


# ----------------------------------------------------------------------------- model
class M1(nn.Module):
    def __init__(self, in_dim: int, d: int = 256, layers: int = 2, heads: int = 4,
                 drop: float = 0.1, n_labels: int = len(LABELS)):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, d), nn.GELU(),
                                 nn.Dropout(drop), nn.Linear(d, d))
        self.kind = nn.Parameter(torch.randn(N_KINDS, 1, d) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, N_SLICES, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, drop, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.queries = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.pool = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True)
        self.norm = nn.LayerNorm(d)
        self.head_w = nn.Parameter(torch.zeros(n_labels, d))
        self.head_b = nn.Parameter(torch.zeros(n_labels))

    def forward(self, x: torch.Tensor, keep: torch.Tensor,
                pos: torch.Tensor | None = None) -> torch.Tensor:
        """x [B, K, S, F] float; keep [B, K, S] bool (True = real token);
        pos [B, S] slice positions 0..31 (None = all 32 slices in order)."""
        b, k, s, _ = x.shape
        pos_emb = self.pos[None] if pos is None else self.pos[0][pos][:, None]   # [.,1,S,d]
        h = self.inp(x) + self.kind[None] + pos_emb                   # [B,K,S,d]
        h = h.reshape(b, k * s, -1)
        pad = ~keep.reshape(b, k * s)
        h = self.enc(h, src_key_padding_mask=pad)
        w = keep.reshape(b, k * s, 1).to(h.dtype)
        h_mean = (h * w).sum(1) / w.sum(1).clamp(min=1.0)                # [B,d]
        q = self.queries[None].expand(b, -1, -1)
        o, _ = self.pool(q, h, h, key_padding_mask=pad)
        o = self.norm(o + h_mean[:, None])     # mean path: learns fast; attention refines
        return (o * self.head_w).sum(-1) + self.head_b                 # [B,12]


# ----------------------------------------------------------------------------- batches
class Features:
    """Holds the cache on the CPU and assembles batches (with optional mirroring)."""

    def __init__(self, cache: dict, use_grid: bool, stats: dict):
        self.stats = stats
        self.f_mean = torch.from_numpy(stats["f_mean"])
        self.f_std = torch.from_numpy(stats["f_std"])
        self.g_mean = torch.from_numpy(stats["g_mean"])
        self.g_std = torch.from_numpy(stats["g_std"])
        self.feats = torch.from_numpy(np.ascontiguousarray(cache["feats"]))
        self.grid = torch.from_numpy(np.ascontiguousarray(cache["grid"])) if use_grid else None
        self.present = torch.from_numpy(cache["present"])
        self.slot = torch.from_numpy(KIND_SLOT)
        self.slot_m = torch.from_numpy(KIND_SLOT_MIRROR)
        self.sag = torch.from_numpy(SAGITTAL)
        self.in_dim = self.feats.shape[-1] + (self.grid.shape[-2] * self.grid.shape[-1]
                                              if use_grid else 0)

    def batch(self, rows: torch.Tensor, mirror: torch.Tensor):
        """rows [B] cache rows, mirror [B] bool -> x [B,4,32,F] float32, present [B,4] bool."""
        slots = torch.where(mirror[:, None], self.slot_m[None], self.slot[None])   # [B,4]
        r = rows[:, None].expand(-1, N_KINDS)
        x = (self.feats[r, slots].float() - self.f_mean) / self.f_std
        if self.grid is not None:
            g = (self.grid[r, slots].float() - self.g_mean) / self.g_std     # [B,4,32,c,384]
            x = torch.cat([x, g.flatten(-2)], -1)
        present = self.present[r, slots]
        flip = (mirror[:, None] & self.sag[None])[:, :, None, None]
        x = torch.where(flip, x.flip(2), x)
        return x, present


def token_mask(present: torch.Tensor, train: bool, p_series: float, p_slice: float) -> torch.Tensor:
    """[B,4] kinds present -> [B,4,32] tokens kept (dropout only when training)."""
    kinds = present.clone()
    if train and p_series > 0:
        drop = (torch.rand(kinds.shape) < p_series) & kinds
        survivors = kinds & ~drop
        drop[~survivors.any(1)] = False            # never drop every series
        kinds = kinds & ~drop
    keep = kinds[:, :, None].expand(-1, -1, N_SLICES).clone()
    keep[~kinds.any(1), 2, 0] = True               # no series at all: one dummy token
    if train and p_slice > 0:
        keep &= torch.rand(keep.shape) >= p_slice
        empty = ~keep.flatten(1).any(1)
        keep[empty] = kinds[empty][:, :, None].expand(-1, -1, N_SLICES)
    return keep


@torch.no_grad()
def predict(model: M1, feats: Features, rows: torch.Tensor, device, tta: bool = True,
            bs: int = 256) -> np.ndarray:
    model.eval()
    out = []
    for s in range(0, len(rows), bs):
        r = rows[s:s + bs]
        acc = 0
        views = (False, True) if tta else (False,)
        for m in views:
            x, present = feats.batch(r, torch.full((len(r),), m))
            keep = token_mask(present, False, 0, 0)
            acc = acc + torch.sigmoid(model(x.to(device), keep.to(device))).cpu()
        out.append(acc / len(views))
    return torch.cat(out).numpy()


# ----------------------------------------------------------------------------- training
def lr_lambda(step: int, total: int, warmup: int) -> float:
    if step < warmup:
        return (step + 1) / warmup
    t = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1 + math.cos(math.pi * min(t, 1.0)))


def train_fold(k: int, df: pd.DataFrame, Y: torch.Tensor, rows: torch.Tensor,
               feats: Features, args, device, log) -> tuple[np.ndarray, M1]:
    torch.manual_seed(args.seed + k)
    tr = torch.from_numpy(np.flatnonzero(df["fold"].to_numpy() != k))
    va = torch.from_numpy(np.flatnonzero(df["fold"].to_numpy() == k))
    model = M1(feats.in_dim, args.d, args.layers, args.heads, args.drop).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
    steps_per_epoch = math.ceil(len(tr) / args.bs)
    total = steps_per_epoch * args.epochs
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda st: lr_lambda(st, total, steps_per_epoch))
    src = df["source"].to_numpy()
    for ep in range(1, args.epochs + 1):
        model.train()
        t0, run_loss = time.time(), 0.0
        perm = tr[torch.randperm(len(tr))]
        for s in range(0, len(perm), args.bs):
            idx = perm[s:s + args.bs]
            mirror = torch.rand(len(idx)) < args.p_mirror
            x, present = feats.batch(rows[idx], mirror)
            keep = token_mask(present, True, args.p_series, args.p_slice)
            logits = model(x.to(device), keep.to(device))
            loss = F.binary_cross_entropy_with_logits(logits, Y[idx].to(device))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            run_loss += loss.item() * len(idx)
        if ep % args.eval_every == 0 or ep == args.epochs:
            p = predict(model, feats, rows[va], device, tta=True)
            sc = score(p, Y[va].numpy(), src[va.numpy()])
            log.info("fold %d ep %2d  loss %.4f  val auc_llm %.4f  auc_human %.3f  (%.1fs)",
                     k, ep, run_loss / len(tr), sc.loc["MACRO", "auc_llm"],
                     sc.loc["MACRO", "auc_human"], time.time() - t0)
    return predict(model, feats, rows[va], device, tta=True), model


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", type=Path, default=Path("."))
    ap.add_argument("--tag", default="dinov2s_224_g4")
    ap.add_argument("--cache", default="cache_g2")
    ap.add_argument("--name", default=None)
    ap.add_argument("--folds", default="0,1,2,3,4")
    ap.add_argument("--no-grid", action="store_true", help="use only CLS+mean features")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--d", type=int, default=256)
    ap.add_argument("--layers", type=int, default=2)
    ap.add_argument("--heads", type=int, default=4)
    ap.add_argument("--drop", type=float, default=0.1)
    ap.add_argument("--p-mirror", type=float, default=0.5)
    ap.add_argument("--p-series", type=float, default=0.2)
    ap.add_argument("--p-slice", type=float, default=0.1)
    ap.add_argument("--eval-every", type=int, default=5)
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--smoke", action="store_true")
    args = ap.parse_args()
    if args.smoke:
        args.folds, args.epochs, args.eval_every = "0", 2, 1
        args.name = args.name or "smoke"

    name = args.name or time.strftime("%Y%m%d_%H%M")
    out = args.root / "runs" / "m1" / name
    out.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s",
                        handlers=[logging.FileHandler(out / "train.log", encoding="utf-8"),
                                  logging.StreamHandler()], force=True)
    log = logging.getLogger("m1")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    labels = read_labels(args.root / "data" / "labels" / "soft_labels.parquet")
    folds = pd.read_csv(args.root / "data" / "labels" / "folds.csv", dtype={"StudyInstanceUID": str})
    cache_dir = args.root / "data" / "embeddings" / args.tag / args.cache
    cache = load_cache(cache_dir)
    stats = load_stats(cache_dir, cache)
    row_of = {u: i for i, u in enumerate(cache["uids"])}
    df = folds.merge(labels[["StudyInstanceUID", *LABELS]], on="StudyInstanceUID")
    df["row"] = df["StudyInstanceUID"].map(row_of)
    missing = df["row"].isna().sum()
    if missing:
        log.warning("%d studies have no embeddings and are skipped", missing)
    df = df.dropna(subset=["row"]).reset_index(drop=True)
    if args.smoke:
        df = df.groupby("fold").head(80).reset_index(drop=True)

    feats = Features(cache, use_grid=not args.no_grid, stats=stats)
    Y = torch.tensor(df[LABELS].to_numpy(), dtype=torch.float32)
    rows = torch.tensor(df["row"].astype(int).to_numpy())
    cfg = {k: (str(v) if isinstance(v, Path) else v) for k, v in vars(args).items()}
    cfg.update(in_dim=feats.in_dim, n_studies=len(df), device=str(device),
               torch=torch.__version__)
    (out / "config.json").write_text(json.dumps(cfg, indent=2))
    log.info("M1 %s: %d studies, in_dim %d, device %s", name, len(df), feats.in_dim, device)

    oof = pd.DataFrame(index=df.index, columns=LABELS, dtype=float)
    for k in [int(f) for f in args.folds.split(",")]:
        pred, model = train_fold(k, df, Y, rows, feats, args, device, log)
        oof.loc[df.index[df["fold"] == k], LABELS] = pred
        torch.save({"state_dict": model.state_dict(), "config": cfg,
                    "stats": {k2: torch.from_numpy(v) for k2, v in stats.items()}},
                   out / f"fold{k}.pt")

    done = oof.notna().all(axis=1).to_numpy()
    res = df.loc[done, ["StudyInstanceUID", "fold", "source"]].join(oof.loc[done])
    res.to_csv(out / "oof.csv", index=False)
    sc = score(oof.loc[done].to_numpy(), Y.numpy()[done], df["source"].to_numpy()[done])
    sc.to_csv(out / "metrics.csv")
    with pd.option_context("display.float_format", "{:.4f}".format):
        log.info("\nOOF scores (%d studies):\n%s", done.sum(), sc.to_string())
    log.info("saved -> %s", out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/code/src/models/m2.py
"""M2: fine-tune DINOv2-S end to end, with the M1 aggregator on top.

    python -m src.models.m2 --smoke                      # tiny pipeline check (~2 min)
    python -m src.models.m2 --folds 0 --name ft1         # one fold first: compare with M1 v2
    python -m src.models.m2 --name ft1                   # all 5 folds (overnight)
    python -m src.models.m2 --name ft2 --collect         # score all oof_fold*.csv of a run
    python -m src.models.m2 --name pl --train-labels pseudo_pl1.parquet   # other TRAINING targets

`--train-labels` (a file in data/labels, same columns as soft_labels.parquet, e.g. pseudo-labels)
replaces only the training targets; validation, OOF scores and metrics are always against
soft_labels.parquet, so the numbers stay comparable with every other run.

Starts from the M1 v2 checkpoint of the SAME fold (head + feature standardisation), so
fold k begins at M1's validation score and fine-tuning should only add to it.

Per training step, each study contributes 4 series x 16 slices (every 2nd slice, random
offset) read straight from data/shards; the first `--frozen-blocks` ViT blocks run
without gradients, the rest are trained (autocast: bf16 where the GPU supports it, else
fp16 with gradient scaling, e.g. on a Kaggle T4; `--amp` overrides). Augmentation per series:
small random scale/shift/rotation, brightness/contrast, and the left<->right knee mirror
(coronal/axial images flipped, sagittal slice order reversed). Whole-series dropout.
Validation: all 32 slices, normal + mirrored view averaged (same TTA as M1).

Outputs: runs/m2/<name>/fold{k}.pt (backbone + head + stats), oof_fold{k}.csv,
metrics.csv (when folds finish), config.json, train.log. When only some folds run (e.g. one
process per GPU on Kaggle), the log and metrics get a suffix (train_f3.log, metrics_f3.csv)
so parallel runs don't overwrite each other; `--collect` then scores all folds together.
"""
from __future__ import annotations

import argparse
import json
import logging
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from .data import sample_study
from .folds import read_labels
from .m1 import M1, lr_lambda
from .metrics import LABELS, score

# ----------------------------------------------------------------------------- data
class StudyDataset(Dataset):
    """Reads one study shard -> uint8 [4, S, 256, 256], present [4], slice positions [S]."""

    def __init__(self, shard_dir: Path, uids, targets: np.ndarray, train: bool,
                 n_slices: int = 16, mirror: bool = False, p_mirror: float = 0.5,
                 p_series: float = 0.2, seed: int = 0):
        self.shard_dir, self.uids, self.targets = Path(shard_dir), list(uids), targets
        self.train, self.n_slices, self.mirror = train, n_slices, mirror
        self.p_mirror, self.p_series, self.seed = p_mirror, p_series, seed
        self.epoch = 0

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        rng = np.random.default_rng((self.seed, self.epoch, i)) if self.train else None
        with np.load(self.shard_dir / f"{self.uids[i]}.npz") as z:
            arrays = {k: z[k] for k in z.files}
        vol, present, idx = sample_study(arrays, self.train, rng, self.n_slices, self.mirror,
                                         self.p_mirror, self.p_series)
        return (torch.from_numpy(vol), torch.from_numpy(present), torch.from_numpy(idx),
                torch.from_numpy(self.targets[i].astype(np.float32)))


# ----------------------------------------------------------------------------- model
class M2(nn.Module):
    def __init__(self, head: M1, stats: dict, backbone: str, img_size: int = 224,
                 grid: int = 2, frozen_blocks: int = 8, weights: str | None = None):
        super().__init__()
        import timm
        kw = dict(pretrained=True, num_classes=0, img_size=img_size)
        if weights:
            kw["pretrained_cfg_overlay"] = dict(file=weights)
        self.backbone = timm.create_model(backbone, **kw)
        self.head = head
        self.img_size, self.grid, self.frozen_blocks = img_size, grid, frozen_blocks
        self.n_prefix = getattr(self.backbone, "num_prefix_tokens", 1)
        cfg = self.backbone.pretrained_cfg
        self.register_buffer("img_mean", torch.tensor(cfg["mean"]).view(1, 3, 1, 1))
        self.register_buffer("img_std", torch.tensor(cfg["std"]).view(1, 3, 1, 1))
        for k in ("f_mean", "f_std", "g_mean", "g_std"):
            self.register_buffer(k, torch.as_tensor(np.asarray(stats[k]), dtype=torch.float32))
        for p in self.backbone.parameters():
            p.requires_grad = False
        for blk in self.backbone.blocks[frozen_blocks:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in self.backbone.norm.parameters():
            p.requires_grad = True

    def stem(self, img: torch.Tensor) -> torch.Tensor:
        """uint8 [N,256,256] -> tokens after the FROZEN blocks (no gradients).
        Identical for every fold, so inference runs it once and shares it."""
        m = self.backbone
        x = img.float().div_(255.0).unsqueeze(1)
        x = F.interpolate(x, size=(self.img_size, self.img_size), mode="bilinear",
                          align_corners=False, antialias=True)
        x = (x.expand(-1, 3, -1, -1) - self.img_mean) / self.img_std
        with torch.no_grad():
            x = m.patch_embed(x)
            x = m._pos_embed(x)
            x = m.patch_drop(x)
            x = m.norm_pre(x)
            for blk in m.blocks[:self.frozen_blocks]:
                x = blk(x)
        return x

    def tail(self, x: torch.Tensor) -> torch.Tensor:
        """stem tokens -> trained blocks -> standardised [N, 768 + g*g*384] (same as M1)."""
        m = self.backbone
        for blk in m.blocks[self.frozen_blocks:]:
            x = blk(x)
        tok = m.norm(x).float()
        cls, patches = tok[:, 0], tok[:, self.n_prefix:]
        n, p, d = patches.shape
        side = int(round(p ** 0.5))
        g = patches.transpose(1, 2).reshape(n, d, side, side)
        g = F.adaptive_avg_pool2d(g, self.grid).flatten(2).transpose(1, 2)      # [N,g*g,d]
        f = (torch.cat([cls, patches.mean(1)], 1) - self.f_mean) / self.f_std
        g = (g - self.g_mean) / self.g_std
        return torch.cat([f, g.flatten(1)], 1)

    def slice_features(self, img: torch.Tensor) -> torch.Tensor:
        return self.tail(self.stem(img))

    def head_from_features(self, feats: torch.Tensor, present: torch.Tensor,
                           pos: torch.Tensor) -> torch.Tensor:
        """feats [n_present_series * S, F] in vol[present] order -> logits [B, 12]."""
        b, k = present.shape
        s = pos.shape[1]
        x = feats.new_zeros(b, k, s, feats.shape[-1])
        x[present] = feats.reshape(-1, s, feats.shape[-1])
        keep = present[:, :, None].expand(-1, -1, s)
        return self.head(x, keep, pos)

    def forward(self, vol: torch.Tensor, present: torch.Tensor, pos: torch.Tensor) -> torch.Tensor:
        b, k, s, h, w = vol.shape
        feats = self.slice_features(vol[present].reshape(-1, h, w))        # only real series
        return self.head_from_features(feats, present, pos)


# ----------------------------------------------------------------------------- loops
def amp_dtype(choice: str, device) -> torch.dtype:
    """auto: bf16 if the GPU has it natively (RTX 30xx+), else fp16 (T4, P100)."""
    if choice == "bf16":
        return torch.bfloat16
    if choice == "fp16" or device.type != "cuda":
        return torch.float16
    try:
        ok = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError:                                   # older torch: no emulation flag
        ok = torch.cuda.get_device_capability()[0] >= 8
    return torch.bfloat16 if ok else torch.float16


def run_eval(model: M2, ds_plain: StudyDataset, ds_mirror: StudyDataset, device,
             workers: int, dtype: torch.dtype = torch.bfloat16) -> np.ndarray:
    model.eval()
    preds = []
    for ds in (ds_plain, ds_mirror):
        out = []
        dl = DataLoader(ds, batch_size=1, num_workers=workers, pin_memory=True)
        with torch.no_grad(), torch.autocast("cuda", dtype=dtype,
                                             enabled=device.type == "cuda"):
            for vol, present, pos, _ in dl:
                logit = model(vol.to(device, non_blocking=True), present.to(device),
                              pos.to(device))
                out.append(torch.sigmoid(logit.float()).cpu())
        preds.append(torch.cat(out).numpy())
    return np.mean(preds, 0)


def train_fold(k, df, Y, args, device, log, out_dir, Yt=None):
    tr = np.flatnonzero(df["fold"].to_numpy() != k)
    va = np.flatnonzero(df["fold"].to_numpy() == k)
    uids = df["StudyInstanceUID"].to_numpy()
    shard_dir = args.root / "data" / "shards"

    ck = torch.load(args.root / "runs" / "m1" / args.init / f"fold{k}.pt",
                    map_location="cpu", weights_only=False)
    cfg1 = ck["config"]
    if cfg1.get("no_grid") or cfg1.get("cache") != "cache_g2":
        raise SystemExit("M2 expects an M1 run trained with the 2x2 grid (cache_g2)")
    head = M1(cfg1["in_dim"], cfg1["d"], cfg1["layers"], cfg1["heads"], args.drop)
    head.load_state_dict(ck["state_dict"])
    stats = {kk: v.numpy() for kk, v in ck["stats"].items()}
    torch.manual_seed(args.seed + k)
    model = M2(head, stats, args.backbone, args.img_size, 2, args.frozen_blocks,
               args.weights).to(device)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info("fold %d: train %d / val %d studies, %.1fM trainable params (head init from "
             "m1/%s)", k, len(tr), len(va), n_train / 1e6, args.init)

    bb = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone")]
    hd = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("head")]
    opt = torch.optim.AdamW([{"params": bb, "lr": args.lr_backbone},
                             {"params": hd, "lr": args.lr_head}], weight_decay=args.wd)
    Yt = Y if Yt is None else Yt                       # training targets (pseudo-labels or Y)
    ds_tr = StudyDataset(shard_dir, uids[tr], Yt[tr], True, args.slices, seed=args.seed + k,
                         p_series=args.p_series)
    dl_tr = DataLoader(ds_tr, batch_size=args.bs, shuffle=True, num_workers=args.workers,
                       pin_memory=True, drop_last=True)
    # not persistent: workers are re-created each epoch so they see ds_tr.epoch (new augmentations)
    ds_va = StudyDataset(shard_dir, uids[va], Y[va], False)
    ds_vm = StudyDataset(shard_dir, uids[va], Y[va], False, mirror=True)
    steps_per_epoch = len(dl_tr) // args.accum
    total = steps_per_epoch * args.epochs
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda st: lr_lambda(st, total, max(1, steps_per_epoch // 2)))
    src = df["source"].to_numpy()
    dtype = amp_dtype(args.amp, device)
    # fp16 needs loss scaling (small gradients underflow); bf16 does not
    scaler = torch.amp.GradScaler("cuda", enabled=dtype == torch.float16 and device.type == "cuda")
    log.info("fold %d: autocast %s%s", k, str(dtype).replace("torch.", ""),
             " + GradScaler" if scaler.is_enabled() else "")

    if args.eval_start:
        p = run_eval(model, ds_va, ds_vm, device, args.workers, dtype)
        sc = score(p, Y[va], src[va])
        log.info("fold %d start (= M1 head)  val auc_llm %.4f  auc_human %.3f", k,
                 sc.loc["MACRO", "auc_llm"], sc.loc["MACRO", "auc_human"])

    for ep in range(1, args.epochs + 1):
        ds_tr.epoch = ep
        model.train()
        model.backbone.blocks[:args.frozen_blocks].eval()      # frozen part: no dropout etc.
        t0, run_loss, n_img, n_seen = time.time(), 0.0, 0, 0
        opt.zero_grad(set_to_none=True)
        for it, (vol, present, pos, y) in enumerate(dl_tr, 1):
            vol, present = vol.to(device, non_blocking=True), present.to(device)
            with torch.autocast("cuda", dtype=dtype, enabled=device.type == "cuda"):
                logit = model(vol, present, pos.to(device))
            loss = F.binary_cross_entropy_with_logits(logit.float(), y.to(device))
            scaler.scale(loss / args.accum).backward()
            run_loss += loss.item() * len(y)
            n_seen += len(y)
            n_img += int(present.sum()) * vol.shape[2]
            if it % args.accum == 0:
                scaler.unscale_(opt)                    # no-op when the scaler is disabled
                nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(opt)                        # skipped if fp16 gradients overflowed
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
            if it % args.log_every == 0:
                el = time.time() - t0
                log.info("  fold %d ep %d  %d/%d  loss %.4f  %.0f img/s  epoch ETA %.1f min",
                         k, ep, it, len(dl_tr), run_loss / n_seen, n_img / el,
                         el / it * (len(dl_tr) - it) / 60)
        msg = f"fold {k} ep {ep}  loss {run_loss / max(n_seen, 1):.4f}  ({(time.time() - t0) / 60:.1f} min)"
        if ep % args.eval_every == 0 or ep == args.epochs:
            p = run_eval(model, ds_va, ds_vm, device, args.workers, dtype)
            sc = score(p, Y[va], src[va])
            msg += (f"  val auc_llm {sc.loc['MACRO', 'auc_llm']:.4f}  "
                    f"auc_human {sc.loc['MACRO', 'auc_human']:.3f}")
        log.info(msg)

    torch.save({"state_dict": model.state_dict(), "m1_config": cfg1, "config": vars_str(args),
                "stats": ck["stats"]}, out_dir / f"fold{k}.pt")
    oof = pd.DataFrame(p, columns=LABELS)
    oof.insert(0, "StudyInstanceUID", uids[va])
    oof.to_csv(out_dir / f"oof_fold{k}.csv", index=False)
    return va, p


def vars_str(args) -> dict:
    return {k: (str(v) if isinstance(v, Path) else v) for k, v in vars(args).items()}


def collect(out_dir: Path, labels: pd.DataFrame, log) -> None:
    """Score every oof_fold*.csv of a run together (after folds ran in separate processes)."""
    parts = sorted(out_dir.glob("oof_fold*.csv"))
    if not parts:
        raise SystemExit(f"no oof_fold*.csv in {out_dir}")
    oof = pd.concat([pd.read_csv(p, dtype={"StudyInstanceUID": str}) for p in parts])
    df = labels[["StudyInstanceUID", "source", *LABELS]].merge(
        oof, on="StudyInstanceUID", suffixes=("", "_p"))
    sc = score(df[[f"{l}_p" for l in LABELS]].to_numpy(), df[LABELS].to_numpy(),
               df["source"].to_numpy())
    sc.to_csv(out_dir / "metrics.csv")
    folds = ",".join(p.stem.replace("oof_fold", "") for p in parts)
    with pd.option_context("display.float_format", "{:.4f}".format):
        log.info("\nOOF scores (%d studies, folds %s):\n%s", len(df), folds, sc.to_string())
    log.info("saved -> %s", out_dir / "metrics.csv")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", type=Path, default=Path("."))
    ap.add_argument("--name", default=None)
    ap.add_argument("--init", default="v2", help="M1 run to initialise the head from")
    ap.add_argument("--train-labels", default=None,
                    help="training targets from this file in data/labels (validation still "
                         "uses soft_labels.parquet)")
    ap.add_argument("--folds", default="0,1,2,3,4")
    ap.add_argument("--backbone", default="vit_small_patch14_dinov2.lvd142m")
    ap.add_argument("--weights", default=None, help="local DINOv2 weights file (optional)")
    ap.add_argument("--img-size", type=int, default=224)
    ap.add_argument("--frozen-blocks", type=int, default=8, help="of 12; the rest are trained")
    ap.add_argument("--slices", type=int, default=16, help="slices per series in training")
    ap.add_argument("--epochs", type=int, default=6)
    ap.add_argument("--bs", type=int, default=2, help="studies per forward pass")
    ap.add_argument("--accum", type=int, default=8, help="gradient accumulation steps")
    ap.add_argument("--lr-backbone", type=float, default=2e-5)
    ap.add_argument("--lr-head", type=float, default=1e-4)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--drop", type=float, default=0.1)
    ap.add_argument("--p-series", type=float, default=0.2)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--eval-every", type=int, default=2)
    ap.add_argument("--eval-start", action="store_true",
                    help="score the untouched start (should match M1 v2 on this fold)")
    ap.add_argument("--log-every", type=int, default=200)
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--amp", default="auto", choices=["auto", "bf16", "fp16"],
                    help="autocast precision; auto = bf16 if native, else fp16 + GradScaler")
    ap.add_argument("--collect", action="store_true",
                    help="only score all oof_fold*.csv in runs/m2/<name> -> metrics.csv")
    ap.add_argument("--smoke", action="store_true")
    args = ap.parse_args()
    if args.smoke:
        args.name, args.folds, args.epochs, args.slices = args.name or "smoke", "0", 1, 8
        args.eval_every, args.log_every, args.accum = 1, 5, 2

    if args.collect and not args.name:
        raise SystemExit("--collect needs --name")
    name = args.name or time.strftime("%Y%m%d_%H%M")
    out_dir = args.root / "runs" / "m2" / name
    out_dir.mkdir(parents=True, exist_ok=True)
    fold_list = [int(f) for f in args.folds.split(",")]
    # a subset of folds (e.g. one process per GPU) gets its own log / metrics file
    sfx = "" if args.collect or sorted(fold_list) == [0, 1, 2, 3, 4] else \
        "_f" + "".join(str(f) for f in fold_list)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s",
                        handlers=[logging.FileHandler(out_dir / f"train{sfx}.log", encoding="utf-8"),
                                  logging.StreamHandler()], force=True)
    log = logging.getLogger("m2")
    labels = read_labels(args.root / "data" / "labels" / "soft_labels.parquet")
    if args.collect:
        collect(out_dir, labels, log)
        return
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    (out_dir / f"config{sfx}.json").write_text(json.dumps(vars_str(args), indent=2))

    folds = pd.read_csv(args.root / "data" / "labels" / "folds.csv",
                        dtype={"StudyInstanceUID": str})
    df = folds.merge(labels[["StudyInstanceUID", *LABELS]], on="StudyInstanceUID")
    have = {p.stem for p in (args.root / "data" / "shards").glob("*.npz")}
    df = df[df["StudyInstanceUID"].isin(have)].reset_index(drop=True)
    if args.smoke:
        df = pd.concat([df[df["fold"] != 0].head(40), df[df["fold"] == 0].head(24)],
                       ignore_index=True)
    Y = df[LABELS].to_numpy().astype(np.float32)
    log.info("M2 %s: %d studies, device %s, torch %s", name, len(df), device, torch.__version__)
    Yt = None
    if args.train_labels:
        tl = Path(args.train_labels)
        tl = tl if tl.exists() else args.root / "data" / "labels" / tl
        pl = read_labels(tl).drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID")
        miss = ~df["StudyInstanceUID"].isin(pl.index)
        if miss.any():
            raise SystemExit(f"{tl.name}: {int(miss.sum())} training studies have no targets")
        Yt = pl.loc[df["StudyInstanceUID"], LABELS].to_numpy().astype(np.float32)
        if not np.isfinite(Yt).all() or Yt.min() < 0 or Yt.max() > 1:
            raise SystemExit(f"{tl.name}: targets must be finite and within [0, 1]")
        log.info("training targets from %s (validation still vs soft_labels.parquet): "
                 "mean |change| %.3f, per label %s", tl.name, float(np.abs(Yt - Y).mean()),
                 ", ".join(f"{l} {d:.3f}" for l, d in zip(LABELS, np.abs(Yt - Y).mean(0))))

    oof = np.full(Y.shape, np.nan)
    for k in fold_list:
        va, p = train_fold(k, df, Y, args, device, log, out_dir, Yt)
        oof[va] = p
    done = ~np.isnan(oof).any(1)
    sc = score(oof[done], Y[done], df["source"].to_numpy()[done])
    sc.to_csv(out_dir / f"metrics{sfx}.csv")
    with pd.option_context("display.float_format", "{:.4f}".format):
        log.info("\nOOF scores (%d studies, folds %s):\n%s", done.sum(), args.folds, sc.to_string())
    log.info("saved -> %s", out_dir)


if __name__ == "__main__":
    main()


## Train

In [ ]:
import shlex, signal, threading, re
from datetime import datetime

TASKS = [(run, k, f"{COMMON} {extra}".strip()) for run, folds, extra in JOBS for k in folds]
NAMES = [j[0] for j in JOBS]
assert TASKS, "JOBS is empty"
assert len(set(NAMES)) == len(NAMES), "each job needs its own run name"

def flag(args, name, default):
    fl = shlex.split(args)
    idx = [i for i, x in enumerate(fl) if x == name]
    return type(default)(fl[idx[-1] + 1]) if idx else default

for run, k, a in TASKS:                          # pseudo-label files must be present
    tl = flag(a, "--train-labels", "")
    assert not tl or os.path.exists(f"{WORK}/data/labels/{tl}"), \
        f"{run}: {tl} not found - attach the rsna-knee-score output that wrote it"
print("tasks:", ", ".join(f"{r} f{k}" for r, k, _ in TASKS))

def run_dir(run):
    d = f"{OUT_ROOT}/{run}"
    os.makedirs(d, exist_ok=True)
    return d

def launch(run, k, args, gpu, extra=()):
    cmd = [sys.executable, "-m", "src.models.m2", "--root", WORK, "--name", run, "--folds", str(k),
           "--weights", WEIGHTS, "--workers", str(WORKERS_PER_GPU), *shlex.split(args), *extra]
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    fh = open(f"{run_dir(run)}/stdout_f{k}.txt", "w")
    print(f"[{time.strftime('%H:%M')}] {run} fold {k} -> GPU {gpu}", flush=True)
    return subprocess.Popen(cmd, cwd=CODE, env=env, stdout=fh, stderr=subprocess.STDOUT,
                            start_new_session=True), fh

def read_log(run, k):
    p = f"{OUT_ROOT}/{run}/train_f{k}.log"
    return open(p, encoding="utf-8").read() if os.path.exists(p) else ""

def tail(run, k):
    lines = [l.rstrip() for l in read_log(run, k).splitlines() if l.strip()]
    return lines[-1] if lines else "(no log yet)"

def gpu_line():
    r = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total,utilization.gpu",
                        "--format=csv,noheader"], capture_output=True, text=True)
    return " | ".join(l.strip() for l in r.stdout.strip().splitlines())

def elapsed_h():
    return (time.time() - T_START) / 3600

PROG = re.compile(r"^(\d{4}-\d\d-\d\d \d\d:\d\d:\d\d),\d+\s+fold \d+ ep 1\s+(\d+)/(\d+)\s+loss [\d.]+"
                  r"\s+(\d+) img/s\s+epoch ETA ([\d.]+) min", re.M)

def epoch_minutes(log):
    hits = PROG.findall(log)
    if not hits:
        return None, None
    ts = [datetime.strptime(h[0], "%Y-%m-%d %H:%M:%S") for h in hits]
    total, ips = int(hits[-1][2]), hits[-1][3]
    if len(hits) >= 2 and int(hits[-1][1]) > int(hits[0][1]):    # time per step between two lines
        per_step = (ts[-1] - ts[0]).total_seconds() / (int(hits[-1][1]) - int(hits[0][1]))
        return per_step * total / 60, ips
    frac = int(hits[-1][1]) / total
    return float(hits[-1][4]) / max(1 - frac, 0.05), ips

if MODE == "benchmark":
    bench = [(f"bench_{run}_f{k}", run, k, a) for run, k, a in TASKS[:len(GPUS)]]
    procs = [launch(b, k, a, g, ["--epochs", "1", "--log-every", "25"])
             for (b, run, k, a), g in zip(bench, GPUS)]
    for _ in range(BENCH_MIN):
        time.sleep(60)
        print(f"[{time.strftime('%H:%M')}] GPU {gpu_line()}", flush=True)
    for p, fh in procs:
        if p.poll() is None:
            os.killpg(p.pid, signal.SIGTERM)
        p.wait(); fh.close()
    fold_h = {}
    for (b, run, k, a), g in zip(bench, GPUS):
        log = read_log(b, k)
        ep_min, ips = epoch_minutes(log)
        if ep_min is None:
            print(f"{run} fold {k}: no progress lines - output:")
            print(open(f"{OUT_ROOT}/{b}/stdout_f{k}.txt").read()[-3000:])
            continue
        n_ep = flag(a, "--epochs", 6)
        fold_h[run] = ep_min * n_ep * 1.15 / 60          # +15% for validation passes
        amp = re.findall(r"autocast.*", log)
        print(f"{run} fold {k} (GPU {g}): {ips} img/s, ~{ep_min:.1f} min/epoch x {n_ep} epochs "
              f"-> ~{fold_h[run]:.1f} h per fold | {amp[0] if amp else ''}")
        shutil.rmtree(f"{OUT_ROOT}/{b}", ignore_errors=True)
    if fold_h:
        slowest = max(fold_h.values())
        free = [0.0] * len(GPUS)
        for run, k, a in TASKS:                          # same order as the real queue
            g = free.index(min(free))
            free[g] += fold_h.get(run, slowest)
        total = max(free) + 0.1
        guessed = sorted({r for r, _, _ in TASKS} - set(fold_h))
        print(f"\nestimate for MODE='train': ~{total:.1f} h for {len(TASKS)} task(s) on {len(GPUS)} GPU(s) "
              f"(budget {TIME_BUDGET_H} h)" + (f"; {guessed} not measured, assumed as slow as the "
              f"slowest measured job" if guessed else ""))
        if total > TIME_BUDGET_H:
            print("-> too long for one session: tasks that would end after the budget are not started; "
                  "move some to the next session's notebook")
else:
    queue, lock, durations, skipped, results = list(TASKS), threading.Lock(), {}, [], []
    ended = set()                                    # (run, fold) finished or crashed

    def estimate(run):
        own = durations.get(run, [])
        anyrun = [h for v in durations.values() for h in v]
        return max(own or anyrun) if (own or anyrun) else None

    def worker(gpu):
        while True:
            with lock:
                task = None
                while queue:
                    run, k, a = queue.pop(0)
                    est = estimate(run)
                    if est is not None and elapsed_h() + est > TIME_BUDGET_H:
                        skipped.append((run, k))
                        continue
                    task = (run, k, a)
                    break
                if task is None:
                    return
            run, k, a = task
            t0 = time.time()
            p, fh = launch(run, k, a, gpu)
            rc = p.wait(); fh.close()
            ended.add((run, k))
            h = (time.time() - t0) / 3600
            vals = re.findall(r"ep (\d+) .*val auc_llm ([\d.]+)\s+auc_human ([\d.na]+)", read_log(run, k))
            last = f"ep {vals[-1][0]} val auc_llm {vals[-1][1]} auc_human {vals[-1][2]}" if vals else "no val score"
            print(f"[{time.strftime('%H:%M')}] {run} fold {k} finished on GPU {gpu}: exit {rc}, "
                  f"{h:.2f} h | {last}", flush=True)
            if rc != 0:
                print(open(f"{OUT_ROOT}/{run}/stdout_f{k}.txt").read()[-3000:], flush=True)
            else:
                with lock:
                    durations.setdefault(run, []).append(h)
                    results.append((run, k, vals))

    def monitor():
        while True:
            time.sleep(MONITOR_MIN * 60)
            print(f"[{time.strftime('%H:%M')}] GPU {gpu_line()}", flush=True)
    threading.Thread(target=monitor, daemon=True).start()

    threads = [threading.Thread(target=worker, args=(g,)) for g in GPUS]
    for t in threads:
        t.start()
    while any(t.is_alive() for t in threads):        # progress in the version's log
        for t in threads:
            t.join(timeout=600 / len(threads))
        for run, k, _ in TASKS:
            if read_log(run, k) and (run, k) not in ended:
                print(f"[{time.strftime('%H:%M')} | {elapsed_h():.1f} h] {run} fold {k}: {tail(run, k)}",
                      flush=True)
    print(f"\nsession finished in {elapsed_h():.1f} h. Validation scores (auc_llm / auc_human):")
    for run, k, vals in sorted(results):
        print(f"  {run:8s} fold {k}: " + "  ".join(f"ep{e} {v} / {h}" for e, v, h in vals))
    if skipped:
        print(f"NOT started (would exceed {TIME_BUDGET_H} h): {skipped} -> put them in the next "
              f"session's JOBS with the same run name and flags")

In [ ]:
if MODE == "train":
    for run in NAMES:
        d = f"{OUT_ROOT}/{run}"
        if not os.path.isdir(d):
            continue
        if len(glob.glob(f"{d}/oof_fold*.csv")) == 5:   # all folds in this session: full OOF score
            subprocess.run([sys.executable, "-m", "src.models.m2", "--root", WORK, "--name", run,
                            "--collect"], cwd=CODE)
        print(f"{run}/")
        for f in sorted(os.listdir(d)):
            print(f"  {f:24s} {os.path.getsize(f'{d}/{f}') / 1e6:8.1f} MB")
else:
    for d in glob.glob(f"{OUT_ROOT}/bench_*"):          # benchmark leaves no output
        shutil.rmtree(d, ignore_errors=True)